In [1]:
# Import library require for process
import pandas as pd
from sklearn.model_selection import train_test_split 
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import r2_score
import warnings as ws
ws.filterwarnings('ignore')



def Forward_Selection(indep_X, dep_Y, n=5):
    models = [
        ("RandomForestRegressor", RandomForestRegressor(n_estimators=10, random_state=42)),
    ]

    backlist = []

    for name, model in models:
       # SequentialFeatureSelector wraps the model and repeatedly tries adding
        # one feature at a time, keeping whichever addition improves the
        # cross-validated score the most, until `n` features are selected.
        sfs = SequentialFeatureSelector(
            model,
            n_features_to_select=n,
            direction='backward',
            scoring='r2',
            cv=5
        )

        # Fits the forward selection process using your features indep_X and target dep_Y.
        sfs.fit(indep_X, dep_Y)

        # get_support() returns a boolean mask of which columns were selected.
        selected_cols = indep_X.columns[sfs.get_support()].tolist()

        # Creates a new DataFrame with only the top selected features.
        selected_features = indep_X[selected_cols]
        backlist.append((name, selected_features))
    return backlist

  
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# r2_prediction - used for regression method, model prediction evaluate method
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
    
# Linear method is used for Linear regression model creation and r2 prediction
def fit_linear(X_train,y_train,X_test):       
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   

# Ridge method is used for svm model creation and r2 prediction
def fit_ridge(X_train,y_train,X_test):                
        regressor = Ridge()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
# Lasso method is used for svm_NL model creation and r2 prediction   
def fit_lasso(X_train,y_train,X_test):                
        regressor = Lasso(alpha=0.1)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# Decision method is used for Decision tree model creation and r2 prediction   
def fit_decision(X_train,y_train,X_test):
        regressor = DecisionTreeRegressor(random_state=0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# SVR method is used for random forest model creation and r2 prediction  
def fit_svr(X_train,y_train,X_test):       
        regressor = SVR(kernel='linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
# random method is used for random forest model creation and r2 prediction  
def fit_random(X_train,y_train,X_test):       
        regressor = RandomForestRegressor(n_estimators=10, random_state=42)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
# RFE_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def Feature_regression(accrf): 
    
    dataframe=pd.DataFrame(index=[ 'Random'],columns=[ 'Random'])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Random'][idex]=accrf[number]
    return dataframe
    

In [2]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using n feature 
FIMList=Forward_Selection(indep_X,dep_Y,8)      


In [3]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send RFE_regression funtion
# finally the evalution data represent by table view.

accrf =[]
index_labels = []


for name, features_df in FIMList:   
    print(name, features_df.shape)
    index_labels.append(name) 
    
    X_train, X_test, y_train, y_test=split_scalar(features_df,dep_Y)  
    r2_r=fit_random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=Feature_regression(accrf)

RandomForestRegressor (399, 8)


In [4]:
result
# 9

,Random
Random,0.915799
